In [8]:
import os
import pandas as pd
from spectral.io import envi
from spectral.io.envi import read_envi_header
from glob import glob

In [19]:
# file paths
home = r'C:\Users\carroll\Documents\sbgPlant'
ref = os.path.join(home, 'schema')
col = os.path.join(home, 'data', 'col_2018')
raw = os.path.join(col, 'raw')
rdn = os.path.join(raw, 'rdn') # path to where your raw flightlines actually are (this will be in the cluster, for now just using my subset example

out_folder = os.path.join(col, 'out_csv')

table = 'sensor_campaign'

In [42]:
# load and view relevant schema and dtype for the table
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# fix typos
schema['column_name'] = schema['column_name'].replace('wavelenght_center', 'wavelength_center')

schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
85,sensor_campaign,sensor_camp_id,character
86,sensor_campaign,band_number,integer
87,sensor_campaign,wavelength_center,double precision
88,sensor_campaign,fwhm,double precision


In [21]:
# get spectral band data directly from any rdn flightline

fp = glob(rdn+'/*.hdr')[0]
hdr = read_envi_header(fp)

In [43]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique(), index=range(len(hdr['wavelength'])))

out_table['sensor_camp_id'] = 'NEON Imaging Spectrometer' # ?
out_table['band_number'] = range(len(hdr['wavelength'])) # 0 or 1 indexed?
out_table['wavelength_center'] = hdr['wavelength']
out_table['fwhm'] = hdr['fwhm']

out_table

,sensor_camp_id,band_number,wavelength_center,fwhm
0,NEON Imaging Spectrometer,0,383.566803,5.617720
1,NEON Imaging Spectrometer,1,388.574707,5.615748
2,NEON Imaging Spectrometer,2,393.582703,5.613800
3,NEON Imaging Spectrometer,3,398.590698,5.611873
4,NEON Imaging Spectrometer,4,403.598694,5.609968
...,...,...,...,...
421,NEON Imaging Spectrometer,421,2491.926758,6.750249
422,NEON Imaging Spectrometer,422,2496.934814,6.757623
423,NEON Imaging Spectrometer,423,2501.942871,6.765019
424,NEON Imaging Spectrometer,424,2506.950684,6.772437


In [44]:
# check data types
out_table.dtypes

sensor_camp_id       object
band_number           int64
wavelength_center    object
fwhm                 object
dtype: object

In [45]:
# update dtypes as necessary

out_table['wavelength_center'] = out_table['wavelength_center'].astype('float')
out_table['fwhm'] = out_table['fwhm'].astype('float')

out_table.dtypes

sensor_camp_id        object
band_number            int64
wavelength_center    float64
fwhm                 float64
dtype: object

In [46]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)